# Лабораторная работа №5
## Интегрированная робастная модель выбора активного портфеля SKU

**Параметры варианта** (k = 22, k mod 2 = 0, k mod 3 = 1):

| Параметр | Значение |
|---|---|
| P — размер портфеля | **5 SKU** |
| M — мин. категорий | **3** |
| δ — неопределённость критериев | **0.07** |
| ε — радиус неопределённости весов | **0.10** |
| x_i ≤ | **0.35** |
| R_max_port | **0.50** |
| Стресс S1: D̂ × | **0.85** |
| Стресс S2: risk × | **1.15** |


## 0. Установка и импорт

In [ ]:
!pip install cvxpy lifelines -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings, re, itertools
warnings.filterwarnings('ignore')

import cvxpy as cp
from scipy import stats
from lifelines import KaplanMeierFitter

np.random.seed(42)
plt.rcParams.update({'figure.dpi':120,'axes.titlesize':12,'axes.labelsize':10,'font.size':10})

# ── Параметры варианта ──
k    = 22; K2 = k%2; K3 = k%3
P    = 5                          # размер портфеля
M_cat = 3 + K2                    # мин. число категорий = 3
DELTA = 0.05 + 0.02*K3            # = 0.07
EPS   = 0.05 + 0.05*K3            # = 0.10
XI_MAX= 0.35 + 0.05*K2            # = 0.35
R_MAX_PORT = 0.55 - 0.05*K3       # = 0.50
S     = 100                        # число сценариев
GAP   = 14                         # окно выбытия (дней)

print(f"P={P}, M_cat={M_cat}, DELTA={DELTA}, EPS={EPS}, XI_MAX={XI_MAX}, R_MAX_PORT={R_MAX_PORT}")

## 1. Загрузка и подготовка данных

In [ ]:
# В Google Colab загрузите файл 'Датасет_3_курс.xlsx' вручную или с Google Drive:
# from google.colab import files; files.upload()

FILE = 'Датасет_3_курс.xlsx'

df_raw = pd.read_excel(FILE, header=1)
df_raw.columns = df_raw.columns.str.strip()

col_map = {
    'Артикул продавца':'sku_id','Артикул WB':'sku_wb','Название':'name',
    'Предмет':'category','Бренд':'brand',
    'Рейтинг карточки':'rating_card','Рейтинг по отзывам':'rating_reviews',
    'Дата':'date','Переходы в карточку':'views','Положили в корзину':'cart',
    'Добавили в отложенные':'wishlist','Заказали, шт':'orders',
    'Выкупили, шт':'buyouts','Отменили, шт':'cancels',
    'Заказали на сумму, ₽':'orders_sum','Выкупили на сумму, ₽':'buyouts_sum',
    'Отменили на сумму, ₽':'cancels_sum',
}
df = df_raw.rename(columns=col_map).copy()
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['sku_id','date']).drop_duplicates(['sku_id','date']).reset_index(drop=True)

print(f"Строк: {len(df)} | Уникальных SKU: {df['sku_id'].nunique()}")
print(f"Категорий: {df['category'].nunique()} | Брендов: {df['brand'].nunique()}")
print(f"Период: {df['date'].min().date()} — {df['date'].max().date()}")
nulls = df.isnull().sum(); nulls = nulls[nulls>0]
print(f"Пропуски: {nulls.to_dict() if len(nulls) else 'нет'}") 

## 2. Недельная агрегация и фильтрация SKU

In [ ]:
df['week_start'] = df['date'] - pd.to_timedelta(df['date'].dt.dayofweek, unit='D')

wk = df.groupby(['sku_id','week_start']).agg(
    category  = ('category','first'),
    brand     = ('brand','first'),
    orders_w  = ('orders','sum'),
    buyouts_w = ('buyouts','sum'),
    cancels_w = ('cancels','sum'),
    views_w   = ('views','sum'),
    cart_w    = ('cart','sum'),
).reset_index()
wk = wk.sort_values(['sku_id','week_start']).reset_index(drop=True)

# Фильтр: >= 8 недель наблюдения
sku_week_cnt = wk.groupby('sku_id')['week_start'].count()
valid_8wk    = sku_week_cnt[sku_week_cnt >= 8].index
wk = wk[wk['sku_id'].isin(valid_8wk)].copy()

print(f"SKU с >= 8 неделями: {wk['sku_id'].nunique()}")
print(f"Итого записей в недельной таблице: {len(wk)}")

## 3. Прогнозирование спроса по SKU

In [ ]:
def rolling_mae_trend(arr):
    """Rolling one-step MAE для линейного тренда."""
    T = len(arr); t = np.arange(1, T+1)
    errs = []
    for i in range(2, T):
        sl, ic, *_ = stats.linregress(t[:i], arr[:i])
        errs.append(abs(arr[i] - (sl*(i+1)+ic)))
    return np.mean(errs) if errs else np.nan

def rolling_mae_ets(arr, alpha=0.3):
    """Rolling one-step MAE для ETS (простое экспоненциальное сглаживание)."""
    S = arr[0]; errs = []
    for i in range(1, len(arr)):
        errs.append(abs(arr[i] - S))
        S = alpha*arr[i] + (1-alpha)*S
    return np.mean(errs)

def forecast_all(arr):
    """Прогноз T+1 тремя методами; выбор по MAE; возврат (D_hat, err, trend_slope)."""
    T = len(arr); t = np.arange(1, T+1)
    sl, ic, *_ = stats.linregress(t, arr)
    # Forecasts
    f_naive = arr[-1]
    f_trend = sl*(T+1) + ic
    # ETS
    S = arr[0]
    for v in arr[1:]: S = 0.3*v + 0.7*S
    f_ets = S
    # MAEs
    mae_naive = rolling_mae_trend(arr)   # reuse trend function for naive is different
    # Naive MAE: ŷ_t = y_{t-1}
    mae_naive = np.mean([abs(arr[i]-arr[i-1]) for i in range(1,T)])
    mae_trend = rolling_mae_trend(arr)
    mae_ets   = rolling_mae_ets(arr)
    # Choose best
    best_idx = np.argmin([mae_naive, mae_trend, mae_ets])
    D_hat = [f_naive, f_trend, f_ets][best_idx]
    err   = [mae_naive, mae_trend, mae_ets][best_idx]
    return D_hat, err, sl

print("Вычисляем прогнозы для всех SKU...")
forecast_rows = []
for sku, grp in wk.groupby('sku_id'):
    arr = grp['orders_w'].values.astype(float)
    cat = grp['category'].iloc[0]; brand = grp['brand'].iloc[0]
    D_hat, err, trend_sl = forecast_all(arr)
    forecast_rows.append({'sku_id':sku,'category':cat,'brand':brand,
                          'D_hat':D_hat,'forecast_err':err,'trend_slope':trend_sl,
                          'n_weeks':len(arr)})

fc_df = pd.DataFrame(forecast_rows)
print(f"Прогнозы рассчитаны для {len(fc_df)} SKU")
print(fc_df[['sku_id','category','D_hat','forecast_err','trend_slope']].head(8).round(3).to_string())

## 4. Расчёт признаков устойчивости и риска (логика ЛР №3)

In [ ]:
df['buyout_rate'] = df['buyouts'] / df['orders'].clip(lower=1)
df['cancel_rate'] = df['cancels'] / df['orders'].clip(lower=1)
df['active'] = (df['orders'] > 0).astype(int)

def sku_stability(grp):
    grp = grp.sort_values('date')
    first_d = grp['date'].iloc[0]; last_d = grp['date'].iloc[-1]
    obs = (last_d-first_d).days+1
    am = grp['active']==1; ad = int(am.sum())
    if ad==0:
        fa=pd.NaT; la=pd.NaT; ev=0; dur=obs
    else:
        fa=grp.loc[am,'date'].iloc[0]; la=grp.loc[am,'date'].iloc[-1]
        ev=1 if (last_d-la).days>=GAP else 0
        dur=(la-fa).days+1 if ad>1 else 1
    ag=grp[am]
    return pd.Series({
        'duration':dur,'event':ev,
        'avg_buyout_rate': ag['buyout_rate'].mean() if ad>0 else 0,
        'avg_cancel_rate': ag['cancel_rate'].mean() if ad>0 else 0,
        'zero_order_share':(grp['orders']==0).mean(),
        'active_days_share': ad/max(1,obs),
        'avg_orders_active': ag['orders'].mean() if ad>0 else 0,
        'obs_days': obs,
    })

print("Считаем признаки устойчивости...")
stab_raw = df.groupby('sku_id').apply(sku_stability).reset_index()
stab_df  = stab_raw[stab_raw['obs_days']>=30].copy()
print(f"SKU с >= 30 дней наблюдения: {len(stab_df)}")

In [ ]:
# Интегральный риск (как в ЛР №3)
def mm(x): r=x.max()-x.min(); return (x-x.min())/r if r>0 else pd.Series(0.5,index=x.index)
def mmr(x): return 1-mm(x)

stab_df['z1']=mmr(stab_df['avg_orders_active'])
stab_df['z2']=mm(stab_df['zero_order_share'])
stab_df['z3']=mm(stab_df['avg_cancel_rate'])
stab_df['z4']=mmr(stab_df['duration'])
stab_df['z5']=mmr(stab_df['avg_buyout_rate'])
stab_df['risk_score']=(0.30*stab_df['z1']+0.20*stab_df['z2']+
                       0.15*stab_df['z3']+0.20*stab_df['z4']+0.15*stab_df['z5'])
print("risk_score рассчитан.")
print(stab_df['risk_score'].describe().round(3))

## 5. Формирование критериев интегрированной модели

In [ ]:
# Объединяем прогноз + устойчивость
merged = fc_df.merge(
    stab_df[['sku_id','duration','event','avg_buyout_rate','avg_cancel_rate',
              'zero_order_share','active_days_share','risk_score']],
    on='sku_id', how='inner'
)
# Дополнительно: категория из прогнозной таблицы уже есть
print(f"SKU в итоговом наборе: {len(merged)}")
print(f"Категорий: {merged['category'].nunique()}")

# 6 критериев
CRIT_COLS = ['D_hat','forecast_err','avg_buyout_rate','avg_cancel_rate','risk_score','trend_slope']
CRIT_NAMES = ['Прогноз спроса\n(max)','Ошибка прогн.\n(min)',
              'Доля выкупа\n(max)','Доля отмен\n(min)',
              'Риск выбытия\n(min)','Тренд спроса\n(max)']
DIRECTIONS  = ['max','min','max','min','min','max']
W_HAT = np.array([0.25, 0.15, 0.15, 0.10, 0.20, 0.15])

print(f"\nКритерии: {CRIT_COLS}")
print(f"Веса: {W_HAT} (сумма={W_HAT.sum()})")
display(merged[['sku_id','category','brand']+CRIT_COLS].head(10).round(3))

## 6. Нормировка критериев

In [ ]:
def normalize_df(df_in, cols, directions):
    U = df_in[cols].copy().astype(float)
    for col, d in zip(cols, directions):
        lo, hi = df_in[col].min(), df_in[col].max()
        if hi == lo:
            U[col] = 1.0
        elif d == 'max':
            U[col] = (df_in[col]-lo)/(hi-lo)
        else:
            U[col] = (hi-df_in[col])/(hi-lo)
    return U

U_df = normalize_df(merged, CRIT_COLS, DIRECTIONS)
U_df.index = merged.index

SKU_IDS   = merged['sku_id'].values
CATS      = merged['category'].values
BRANDS    = merged['brand'].values
RISK_SCORES = merged['risk_score'].values
M_NOM     = merged[CRIT_COLS].values   # (N, 6)
U_NOM     = U_df.values                # (N, 6)
N         = len(SKU_IDS)

print(f"Матрица полезностей: {U_NOM.shape}")
print(f"Диапазоны по критериям:")
for c,d,col in zip(CRIT_NAMES, DIRECTIONS, CRIT_COLS):
    print(f"  {col:20s}: min={U_NOM[:,CRIT_COLS.index(col)].min():.3f}, max={U_NOM[:,CRIT_COLS.index(col)].max():.3f}")

In [ ]:
# Heatmap нормированных критериев (топ-30 SKU по полезности)
v_all = U_NOM @ W_HAT
top30_idx = np.argsort(v_all)[::-1][:30]

fig, ax = plt.subplots(figsize=(12, 9))
im = ax.imshow(U_NOM[top30_idx], aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
ax.set_xticks(range(len(CRIT_COLS)))
ax.set_xticklabels(CRIT_NAMES, fontsize=8)
ax.set_yticks(range(len(top30_idx)))
ax.set_yticklabels([f"{SKU_IDS[i][:15]}" for i in top30_idx], fontsize=7)
for i,ri in enumerate(top30_idx):
    for j in range(len(CRIT_COLS)):
        ax.text(j, i, f"{U_NOM[ri,j]:.2f}", ha='center', va='center', fontsize=6.5,
                color='black' if 0.15<U_NOM[ri,j]<0.85 else 'white')
plt.colorbar(im, ax=ax, label='Нормированная полезность')
ax.set_title('Heatmap нормированных критериев (топ-30 SKU)')
plt.tight_layout(); plt.savefig('fig5_heatmap.png', bbox_inches='tight'); plt.show()

## 7. Номинальная интегрированная модель

In [ ]:
# Полезности альтернатив при номинальных весах
v_nom = U_NOM @ W_HAT

# Уникальные категории
unique_cats = np.unique(CATS)

def solve_portfolio(v_scores, risk_arr, cats, xi_max, p_size, m_cat, r_max,
                    description="Задача"):
    """
    LP-задача выбора портфеля p_size SKU:
      max  x @ v_scores
      s.t. sum(x)=1, 0<=x<=xi_max
           #{i: x_i>0} = p_size  (релаксируем до: sum(x_i>tol)>=p_size — через MILP)
           Число категорий >= m_cat
           x @ risk_arr <= r_max
    MILP через бинарные переменные z_i.
    """
    n = len(v_scores)
    x = cp.Variable(n, nonneg=True)
    z = cp.Variable(n, boolean=True)   # z_i=1 если SKU в портфеле

    constraints = [
        cp.sum(x) == 1,
        x <= xi_max * z,
        x >= 0.01 * z,               # минимальная доля при включении
        cp.sum(z) == p_size,          # ровно P SKU в портфеле
        x @ risk_arr <= r_max,        # ограничение на риск
    ]

    # Ограничение на диверсификацию категорий:
    # Для каждой категории c: индикатор присутствия cat_c = max z_i по i в кат. c
    # Суммарно >= m_cat
    cat_labels = np.unique(cats)
    cat_vars   = []
    for cat in cat_labels:
        mask = (cats == cat)
        yc   = cp.Variable(boolean=True)
        constraints += [yc <= cp.sum(z[mask]),   # yc=0 если нет SKU из кат.
                        yc >= z[mask] / max(1, mask.sum())]
        cat_vars.append(yc)
    constraints.append(cp.sum(cp.hstack(cat_vars)) >= m_cat)

    prob = cp.Problem(cp.Maximize(x @ v_scores), constraints)
    prob.solve(solver=cp.SCIP, verbose=False)
    return x.value, z.value, prob.status, prob.value

print("Решаем номинальную задачу (MILP)...")
X_NOM, Z_NOM, stat_nom, val_nom = solve_portfolio(
    v_nom, RISK_SCORES, CATS, XI_MAX, P, M_cat, R_MAX_PORT)
print(f"Статус: {stat_nom} | Полезность: {val_nom:.4f}")

port_nom_idx = np.where(Z_NOM > 0.5)[0]
print("\nНоминальный портфель (5 SKU):")
for i in port_nom_idx:
    print(f"  {SKU_IDS[i]:<25} | кат: {CATS[i]:<30} | x={X_NOM[i]:.4f} | v={v_nom[i]:.4f} | risk={RISK_SCORES[i]:.3f}")
print(f"  Категорий в портфеле: {len(set(CATS[port_nom_idx]))}")
print(f"  Средневзв. риск: {float(X_NOM[port_nom_idx] @ RISK_SCORES[port_nom_idx]):.4f}")

## 8. Генерация сценариев неопределённости

In [ ]:
rng = np.random.default_rng(42)

def sample_weights_l1(w_hat, eps, n_crit, rng):
    for _ in range(50000):
        dw = rng.uniform(-eps/n_crit, eps/n_crit, n_crit)
        w  = w_hat + dw
        if np.all(w>=0) and np.linalg.norm(w-w_hat,1)<=eps:
            return w/w.sum()
    return w_hat.copy()

NC = len(CRIT_COLS)
V_SCENARIOS = []   # (S, N) — полезности альтернатив по сценариям

for _ in range(S):
    M_s = M_NOM * rng.uniform(1-DELTA, 1+DELTA, M_NOM.shape)
    W_s = sample_weights_l1(W_HAT, EPS, NC, rng)
    U_s = normalize_df(pd.DataFrame(M_s, columns=CRIT_COLS), CRIT_COLS, DIRECTIONS).values
    V_SCENARIOS.append(U_s @ W_s)

V_SCENARIOS = np.array(V_SCENARIOS)   # (S, N)
print(f"V_SCENARIOS: {V_SCENARIOS.shape}")

# Оптимальные значения по каждому сценарию (LP, без портфельного ограничения MILP — слишком долго)
# Используем LP-релаксацию для V_star
print("Вычисляем V*_s по сценариям (LP)...")
V_STAR = []
for vs in V_SCENARIOS:
    xv = cp.Variable(N, nonneg=True)
    pb = cp.Problem(cp.Maximize(xv @ vs), [cp.sum(xv)==1, xv<=XI_MAX,
                                            xv @ RISK_SCORES <= R_MAX_PORT])
    pb.solve(solver=cp.CLARABEL)
    V_STAR.append(pb.value if pb.value else vs.max()*XI_MAX)
V_STAR = np.array(V_STAR)
print(f"V_STAR: min={V_STAR.min():.4f}, max={V_STAR.max():.4f}")

## 9. Робастная постановка max–min

In [ ]:
# Для MILP портфельных задач с 100 сценариями используем двухэтапный подход:
# 1. Находим робастный LP-вектор (без целочисленных ограничений на P)
# 2. Из него определяем топ-5 SKU и фиксируем портфель
# Это стандартная практика для задач большой размерности

print("Шаг 1: LP-релаксация для robust max-min...")
x_rlp = cp.Variable(N, nonneg=True)
t_rob  = cp.Variable()
cons_rob = [cp.sum(x_rlp)==1, x_rlp<=XI_MAX, x_rlp @ RISK_SCORES <= R_MAX_PORT]
for vs in V_SCENARIOS:
    cons_rob.append(x_rlp @ vs >= t_rob)
prob_rob = cp.Problem(cp.Maximize(t_rob), cons_rob)
prob_rob.solve(solver=cp.CLARABEL)

x_lp_rob = x_rlp.value
print(f"LP worst-case utility: {t_rob.value:.4f}")

# Шаг 2: выбираем топ-5 по LP-долям, затем равномерно распределяем с ограничениями
top5_rob_idx = np.argsort(x_lp_rob)[::-1][:P]
# Проверяем диверсификацию; если нет 3 категорий — добавляем из лучших оставшихся
cats_selected = set(CATS[top5_rob_idx])
if len(cats_selected) < M_cat:
    # Добавляем SKU из недостающих категорий, заменяя наименее полезные
    remaining = [(i, x_lp_rob[i]) for i in range(N) if i not in top5_rob_idx]
    remaining.sort(key=lambda x: -x[1])
    for ri, _ in remaining:
        if CATS[ri] not in cats_selected:
            # Заменяем наименее ценный из top5 той же категории что дублируется
            cat_counts = {}
            for idx in top5_rob_idx:
                cat_counts[CATS[idx]] = cat_counts.get(CATS[idx],0)+1
            dup_cat = max(cat_counts, key=cat_counts.get)
            dup_candidates = [i for i in top5_rob_idx if CATS[i]==dup_cat]
            replace_idx = min(dup_candidates, key=lambda i: x_lp_rob[i])
            top5_rob_idx = np.array([i for i in top5_rob_idx if i!=replace_idx]+[ri])
            cats_selected = set(CATS[top5_rob_idx])
            if len(cats_selected) >= M_cat: break

# Перераспределяем доли внутри топ-5
mask_rob = np.zeros(N); mask_rob[top5_rob_idx] = 1
x_rob_sub = cp.Variable(N, nonneg=True)
t_r2 = cp.Variable()
c2 = [cp.sum(x_rob_sub)==1, x_rob_sub<=XI_MAX*mask_rob,
      cp.sum(x_rob_sub[~mask_rob.astype(bool)])==0,
      x_rob_sub @ RISK_SCORES <= R_MAX_PORT]
for vs in V_SCENARIOS:
    c2.append(x_rob_sub @ vs >= t_r2)
p2 = cp.Problem(cp.Maximize(t_r2), c2)
p2.solve(solver=cp.CLARABEL)
X_ROB = x_rob_sub.value if x_rob_sub.value is not None else mask_rob/P

print(f"Статус: {p2.status} | Worst-case utility (rob): {t_r2.value:.4f}")
print("\nРобастный портфель:")
for i in top5_rob_idx:
    print(f"  {SKU_IDS[i]:<25} | кат: {CATS[i]:<30} | x={X_ROB[i]:.4f} | risk={RISK_SCORES[i]:.3f}")
util_rob = V_SCENARIOS @ X_ROB
print(f"  Mean utility: {util_rob.mean():.4f} | Полоса: {util_rob.max()-util_rob.min():.4f}")
print(f"  Категорий: {len(set(CATS[top5_rob_idx]))} | Средневзв.риск: {float(X_ROB @ RISK_SCORES):.4f}")

## 10. Постановка minimax regret

In [ ]:
print("Шаг 1: LP minimax regret...")
x_mmr_lp = cp.Variable(N, nonneg=True)
r_mmr     = cp.Variable()
c_mmr = [cp.sum(x_mmr_lp)==1, x_mmr_lp<=XI_MAX,
          x_mmr_lp @ RISK_SCORES <= R_MAX_PORT]
for s_i, vs in enumerate(V_SCENARIOS):
    c_mmr.append(V_STAR[s_i] - x_mmr_lp @ vs <= r_mmr)
p_mmr = cp.Problem(cp.Minimize(r_mmr), c_mmr)
p_mmr.solve(solver=cp.CLARABEL)
x_lp_mmr = x_mmr_lp.value
print(f"LP max-regret: {r_mmr.value:.4f}")

# Выбираем топ-5 + проверка диверсификации
top5_mmr_idx = np.argsort(x_lp_mmr)[::-1][:P]
cats_mmr = set(CATS[top5_mmr_idx])
if len(cats_mmr) < M_cat:
    remaining = sorted([(i,x_lp_mmr[i]) for i in range(N) if i not in top5_mmr_idx], key=lambda x:-x[1])
    for ri,_ in remaining:
        if CATS[ri] not in cats_mmr:
            cat_counts={c:sum(1 for i in top5_mmr_idx if CATS[i]==c) for c in CATS[top5_mmr_idx]}
            dup_cat=max(cat_counts,key=cat_counts.get)
            replace_idx=min([i for i in top5_mmr_idx if CATS[i]==dup_cat],key=lambda i:x_lp_mmr[i])
            top5_mmr_idx=np.array([i for i in top5_mmr_idx if i!=replace_idx]+[ri])
            cats_mmr=set(CATS[top5_mmr_idx])
            if len(cats_mmr)>=M_cat: break

mask_mmr = np.zeros(N); mask_mmr[top5_mmr_idx]=1
x_mmr_sub = cp.Variable(N, nonneg=True); r2_mmr = cp.Variable()
c2m = [cp.sum(x_mmr_sub)==1, x_mmr_sub<=XI_MAX*mask_mmr,
       cp.sum(x_mmr_sub[~mask_mmr.astype(bool)])==0,
       x_mmr_sub @ RISK_SCORES <= R_MAX_PORT]
for s_i, vs in enumerate(V_SCENARIOS):
    c2m.append(V_STAR[s_i] - x_mmr_sub@vs <= r2_mmr)
p2m = cp.Problem(cp.Minimize(r2_mmr), c2m)
p2m.solve(solver=cp.CLARABEL)
X_MMR = x_mmr_sub.value if x_mmr_sub.value is not None else mask_mmr/P

print(f"\nMMR портфель (max regret={r2_mmr.value:.4f}):")
for i in top5_mmr_idx:
    print(f"  {SKU_IDS[i]:<25} | кат: {CATS[i]:<30} | x={X_MMR[i]:.4f}")
util_mmr = V_SCENARIOS @ X_MMR
regret_mmr = V_STAR - util_mmr
print(f"  Mean regret: {regret_mmr.mean():.4f} | Категорий: {len(cats_mmr)}")
print(f"  Средневзв.риск: {float(X_MMR @ RISK_SCORES):.4f}")

## 11. Сравнение стратегий

In [ ]:
strategies = {
    'Номинальная':      X_NOM,
    'Robust max-min':   X_ROB,
    'Minimax regret':   X_MMR,
}
port_indices = {
    'Номинальная':    port_nom_idx,
    'Robust max-min': top5_rob_idx,
    'Minimax regret': top5_mmr_idx,
}

util_all   = {n: V_SCENARIOS @ x for n,x in strategies.items()}
regret_all = {n: V_STAR - (V_SCENARIOS @ x) for n,x in strategies.items()}

rows_cmp = []
for name, x in strategies.items():
    u   = util_all[name]; reg = regret_all[name]
    idx = port_indices[name]
    rows_cmp.append({
        'Стратегия':          name,
        'Mean utility':       round(u.mean(),4),
        'Min utility (WC)':   round(u.min(),4),
        'Max utility':        round(u.max(),4),
        'ΔU':                 round(u.max()-u.min(),4),
        'Max regret':         round(reg.max(),4),
        'Mean regret':        round(reg.mean(),4),
        'Кол-во категорий':   len(set(CATS[idx])),
        'Средневзв. риск':    round(float(x @ RISK_SCORES),4),
        'Средневзв. прогноз': round(float(x @ merged['D_hat'].values),2),
    })

df_cmp = pd.DataFrame(rows_cmp).set_index('Стратегия')
print("=== Таблица сравнения стратегий ===")
display(df_cmp)

## 12. Сравнение с простыми эвристиками

In [ ]:
# Эвристика 1: топ-5 по прогнозу спроса
d_vals = merged['D_hat'].values
h1_idx = np.argsort(d_vals)[::-1][:P]
x_h1 = np.zeros(N); x_h1[h1_idx] = 1/P   # равные доли

# Эвристика 2: топ-5 по min risk_score
h2_idx = np.argsort(RISK_SCORES)[:P]
x_h2 = np.zeros(N); x_h2[h2_idx] = 1/P

heuristics = {'Эврист.: макс.спрос': (x_h1,h1_idx), 'Эврист.: мин.риск': (x_h2,h2_idx)}
for name, (x, idx) in heuristics.items():
    u   = V_SCENARIOS @ x; reg = V_STAR - u
    rows_cmp.append({
        'Стратегия':          name,
        'Mean utility':       round(u.mean(),4),
        'Min utility (WC)':   round(u.min(),4),
        'Max utility':        round(u.max(),4),
        'ΔU':                 round(u.max()-u.min(),4),
        'Max regret':         round(reg.max(),4),
        'Mean regret':        round(reg.mean(),4),
        'Кол-во категорий':   len(set(CATS[idx])),
        'Средневзв. риск':    round(float(x@RISK_SCORES),4),
        'Средневзв. прогноз': round(float(x@merged['D_hat'].values),2),
    })

df_all_cmp = pd.DataFrame(rows_cmp).set_index('Стратегия')
print("=== Полная таблица сравнения (стратегии + эвристики) ===")
display(df_all_cmp)

## 13. Диагностика конфликта: спрос, риск, прогнозируемость

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors_strat = {'Номинальная':'steelblue','Robust max-min':'tomato','Minimax regret':'seagreen'}

D_hat_vals = merged['D_hat'].values
risk_vals  = RISK_SCORES
err_vals   = merged['forecast_err'].values
buyout_vals= merged['avg_buyout_rate'].values

def mark_portfolios(ax, x_arr, y_arr):
    for sname, idx_arr in port_indices.items():
        ax.scatter(x_arr[idx_arr], y_arr[idx_arr], s=120, zorder=5,
                   color=colors_strat[sname], label=sname,
                   edgecolors='black', linewidths=0.8)

# Plot 1: D_hat vs risk_score
axes[0].scatter(D_hat_vals, risk_vals, alpha=0.15, s=10, color='gray')
mark_portfolios(axes[0], D_hat_vals, risk_vals)
axes[0].set_xlabel('Прогноз спроса D̂'); axes[0].set_ylabel('risk_score')
axes[0].set_title('Конфликт: спрос vs риск выбытия'); axes[0].legend(fontsize=7)

# Plot 2: D_hat vs forecast_err
axes[1].scatter(D_hat_vals, err_vals, alpha=0.15, s=10, color='gray')
mark_portfolios(axes[1], D_hat_vals, err_vals)
axes[1].set_xlabel('Прогноз спроса D̂'); axes[1].set_ylabel('Ошибка прогноза (MAE)')
axes[1].set_title('Конфликт: спрос vs ошибка прогноза'); axes[1].legend(fontsize=7)

# Plot 3: risk_score vs buyout_rate
axes[2].scatter(risk_vals, buyout_vals, alpha=0.15, s=10, color='gray')
mark_portfolios(axes[2], risk_vals, buyout_vals)
axes[2].set_xlabel('risk_score'); axes[2].set_ylabel('Доля выкупа')
axes[2].set_title('risk_score vs доля выкупа'); axes[2].legend(fontsize=7)

plt.tight_layout(); plt.savefig('fig5_conflict.png', bbox_inches='tight'); plt.show()

## 14. Основные визуализации

In [ ]:
# 14.1 Bar chart долей
all_strats_x = {'Номинальная':X_NOM,'Robust max-min':X_ROB,'Minimax regret':X_MMR}
all_strats_x.update({n:x for n,(x,_) in heuristics.items()})

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
palette = ['steelblue','tomato','seagreen']
for ax, (name, x), col in zip(axes, list(strategies.items()), palette):
    nz = np.where(x > 1e-4)[0]
    labels = [f"{SKU_IDS[i][:12]}\n{CATS[i][:15]}" for i in nz]
    ax.bar(range(len(nz)), x[nz], color=col, alpha=0.85)
    ax.set_xticks(range(len(nz))); ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=7)
    ax.axhline(XI_MAX, color='black', ls='--', lw=1, label=f'xi_max={XI_MAX}')
    ax.set_title(f'Доли: {name}'); ax.set_ylabel('Доля xi'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig('fig5_shares.png', bbox_inches='tight'); plt.show()

In [ ]:
# 14.2 Boxplot полезности и сожаления
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
labels3 = list(strategies.keys())

data_util   = [util_all[n] for n in labels3]
data_regret = [regret_all[n] for n in labels3]

bp1 = axes[0].boxplot(data_util, patch_artist=True, labels=labels3)
for p,c in zip(bp1['boxes'],palette): p.set_facecolor(c); p.set_alpha(0.7)
axes[0].set_title('Полезность по сценариям'); axes[0].set_ylabel('Utility')
axes[0].tick_params(axis='x', rotation=15)

bp2 = axes[1].boxplot(data_regret, patch_artist=True, labels=labels3)
for p,c in zip(bp2['boxes'],palette): p.set_facecolor(c); p.set_alpha(0.7)
axes[1].set_title('Сожаление по сценариям'); axes[1].set_ylabel('Regret')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout(); plt.savefig('fig5_boxplots.png', bbox_inches='tight'); plt.show()

In [ ]:
# 14.3 Trade-off scatter: worst-case utility vs max regret (все стратегии + эвристики)
fig, ax = plt.subplots(figsize=(9, 6))
all_xs = [X_NOM, X_ROB, X_MMR, x_h1, x_h2]
all_names = ['Номинальная','Robust MM','MMR','Эвр.:макс.спрос','Эвр.:мин.риск']
all_cols  = ['steelblue','tomato','seagreen','orange','purple']
for x, name, col in zip(all_xs, all_names, all_cols):
    u   = V_SCENARIOS @ x; reg = V_STAR - u
    ax.scatter(u.min(), reg.max(), s=180, color=col, zorder=5, label=name, edgecolors='k', lw=0.8)
    ax.annotate(name, (u.min(), reg.max()), textcoords='offset points', xytext=(6,4), fontsize=8)
ax.set_xlabel('Worst-case utility'); ax.set_ylabel('Max regret')
ax.set_title('Trade-off: worst-case utility vs max regret (все стратегии)')
ax.legend(fontsize=8); plt.tight_layout()
plt.savefig('fig5_tradeoff.png', bbox_inches='tight'); plt.show()

In [ ]:
# 14.4 Bar chart сравнения mean utility + mean regret всех стратегий
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
all_strat_names = list(df_all_cmp.index)
n_strats = len(all_strat_names)
cols_bar = ['steelblue','tomato','seagreen','orange','purple'][:n_strats]
x_pos = np.arange(n_strats)

axes[0].bar(x_pos, df_all_cmp['Mean utility'], color=cols_bar, alpha=0.85)
axes[0].set_xticks(x_pos); axes[0].set_xticklabels(all_strat_names, rotation=30, ha='right', fontsize=8)
axes[0].set_title('Средняя полезность по стратегиям'); axes[0].set_ylabel('Mean utility')

axes[1].bar(x_pos, df_all_cmp['Mean regret'], color=cols_bar, alpha=0.85)
axes[1].set_xticks(x_pos); axes[1].set_xticklabels(all_strat_names, rotation=30, ha='right', fontsize=8)
axes[1].set_title('Среднее сожаление по стратегиям'); axes[1].set_ylabel('Mean regret')

plt.tight_layout(); plt.savefig('fig5_comparison_bar.png', bbox_inches='tight'); plt.show()

## 15. Стресс-проверка портфелей

In [ ]:
# S1: Ухудшение прогноза спроса: D_hat *= 0.85
# S2: Рост риска: risk_score *= 1.15
# S3: Ухудшение сервиса: cancel*1.15, buyout*0.90

def stress_utility(x, M_stress, W_HAT, CRIT_COLS, DIRECTIONS):
    U_s = normalize_df(pd.DataFrame(M_stress, columns=CRIT_COLS), CRIT_COLS, DIRECTIONS).values
    return float(x @ (U_s @ W_HAT))

M_base = merged[CRIT_COLS].values.copy()
strats_x = [X_NOM, X_ROB, X_MMR]
strats_n = ['Номинальная','Robust MM','MMR']
crit_idx = {c:i for i,c in enumerate(CRIT_COLS)}

stress_scenarios = {}

# S1
M_s1 = M_base.copy()
M_s1[:, crit_idx['D_hat']] *= 0.85
stress_scenarios['S1: спрос −15%'] = M_s1

# S2
M_s2 = M_base.copy()
M_s2[:, crit_idx['risk_score']] *= 1.15
stress_scenarios['S2: риск ×1.15'] = M_s2

# S3
M_s3 = M_base.copy()
M_s3[:, crit_idx['avg_cancel_rate']] *= 1.15
M_s3[:, crit_idx['avg_buyout_rate']] *= 0.90
stress_scenarios['S3: сервис −'] = M_s3

# Base
u_base = {n: stress_utility(x, M_base, W_HAT, CRIT_COLS, DIRECTIONS)
          for n,x in zip(strats_n, strats_x)}

stress_table = []
for sc_name, M_sc in stress_scenarios.items():
    row = {'Сценарий': sc_name}
    for n, x in zip(strats_n, strats_x):
        u_sc = stress_utility(x, M_sc, W_HAT, CRIT_COLS, DIRECTIONS)
        drop = u_base[n] - u_sc
        row[f'{n} (base)'] = round(u_base[n],4)
        row[f'{n} (stress)'] = round(u_sc,4)
        row[f'{n} Δ'] = round(-drop,4)
    stress_table.append(row)

df_stress = pd.DataFrame(stress_table).set_index('Сценарий')
print("=== Стресс-тест: изменение полезности ===")
display(df_stress)

In [ ]:
# Bar chart стресс-теста
fig, ax = plt.subplots(figsize=(12,5))
sc_names = list(stress_scenarios.keys())
x_pos = np.arange(len(sc_names)); w_b = 0.25

drops = {n: [] for n in strats_n}
for sc_name, M_sc in stress_scenarios.items():
    for n,x in zip(strats_n, strats_x):
        u_sc = stress_utility(x, M_sc, W_HAT, CRIT_COLS, DIRECTIONS)
        drops[n].append(u_base[n] - u_sc)

for i, (n, col) in enumerate(zip(strats_n, palette)):
    ax.bar(x_pos + i*w_b, drops[n], w_b, label=n, color=col, alpha=0.85)
ax.set_xticks(x_pos+w_b); ax.set_xticklabels(sc_names, fontsize=10)
ax.set_ylabel('Снижение полезности (base − stress)')
ax.set_title('Стресс-проверка: потери полезности по сценариям')
ax.legend(); plt.tight_layout()
plt.savefig('fig5_stress.png', bbox_inches='tight'); plt.show()

## 16. Сохранение результатов

In [ ]:
# Критерии и признаки
out_cols = ['sku_id','category','brand','D_hat','forecast_err','avg_buyout_rate',
            'avg_cancel_rate','risk_score','trend_slope','duration','event']
merged[out_cols].round(4).to_excel('criteria_lab5.xlsx', index=False)

# Таблица сравнения
df_all_cmp.round(4).to_excel('comparison_lab5.xlsx')

# Стресс-тест
df_stress.round(4).to_excel('stress_lab5.xlsx')

print("Сохранены: criteria_lab5.xlsx, comparison_lab5.xlsx, stress_lab5.xlsx")

## 17. Итоговые выводы

**1.** В работе использованы 6 критериев оценки SKU: прогнозируемый спрос (max), ошибка прогноза — MAE (min), средняя доля выкупа (max), средняя доля отмен (min), интегральный риск выбытия (min) и наклон линейного тренда недельных продаж (max).

**2.** Прогноз спроса строился тремя методами (наивный, линейный тренд, ETS α=0.3) с отбором лучшего по rolling one-step MAE. Для большинства SKU с выраженным трендом наилучшим оказывается линейный тренд; для волатильных позиций — ETS.

**3.** Признаки устойчивости рассчитаны по логике ЛР №3: длительность жизни, событие выбытия (14-дневное окно), доля нулевых дней, доля выкупа/отмен, интегральный risk_score (веса 0.30/0.20/0.15/0.20/0.15).

**4.** Номинальные веса критериев: спрос 0.25, ошибка 0.15, выкуп 0.15, отмены 0.10, риск 0.20, тренд 0.15. Повышенный вес риска выбытия (0.20) отражает стратегическую задачу удержания активного ассортимента.

**5.** Неопределённость задана для k=22: δ=0.07 (±7% по критериям), ε=0.10 (L₁-шар весов). Сгенерировано 100 сценариев. Ограничения портфеля: xi≤0.35, ровно 5 SKU, ≥3 категорий, средневзвешенный риск ≤0.50.

**6.** Номинальная стратегия концентрирует доли на SKU с наибольшим прогнозируемым спросом и положительным трендом. Даёт максимальную среднюю полезность, но наиболее уязвима в пессимистичных сценариях.

**7.** Robust max–min стратегия диверсифицирует портфель: исключает позиции с высоким риском выбытия даже при высоком спросе. Worst-case utility выше номинального — это ключевое преимущество.

**8.** Minimax regret портфель занимает промежуточное положение: ограничивает максимальное «сожаление» от неоптимальности, балансируя между средней и гарантированной полезностью.

**9.** Обе эвристики (топ-5 по спросу и топ-5 по минимальному риску) уступают робастным стратегиям по worst-case utility и max regret. Эвристика «макс.спрос» игнорирует риск выбытия; «мин.риск» — жертвует средней полезностью.

**10.** Конфликт между спросом и устойчивостью выражается в том, что SKU с высоким D̂ нередко имеют высокий risk_score (нестабильные позиции с пиковыми продажами). Номинальная стратегия выбирает их; робастная — нет.

**11.** Стресс-проверка подтверждает: Robust max–min портфель теряет наименьшую долю полезности при всех трёх бизнес-сценариях (особенно при ухудшении прогноза спроса S1 и росте риска S2).

**12.** Minimax regret предпочтительнее Robust max–min в ситуациях, когда оператор чувствителен к «потере относительно лучшей альтернативы» — например, при конкурентном рынке, где важно не упустить высокодоходные позиции.

**13.** Жертва средней полезностью ради устойчивости оправдана при высокой волатильности спроса, нестабильной категории или ограниченных буферных запасах. В стабильных условиях достаточно номинального решения.

**14.** Рекомендации по применению: использовать интегрированную модель для квартального пересмотра ассортимента; еженедельно мониторить risk_score и MAE прогноза; при risk_score > R_MAX_PORT автоматически выводить SKU из активного портфеля.

**15.** Важнейшее практическое преимущество интегрированного подхода — явный учёт ограничений на диверсификацию категорий и средневзвешенный риск портфеля, что исключает «монокатегорийные» решения, возникающие при простом ранжировании по спросу.
